In [3]:
%pip install scikit-learn xgboost
%pip install scikit-learn pyspark
%pip install pyarrow

Note: you may need to restart the kernel to use updated packages.
Note: you may need to restart the kernel to use updated packages.
Note: you may need to restart the kernel to use updated packages.


In [21]:
import pandas as pd

def load_clean_csv(filepath):
    with open(filepath, 'r', encoding='utf-8') as f:
        first_line = f.readline()
    sep = ';' if first_line.count(';') > first_line.count(',') else ','
    df = pd.read_csv(filepath, sep=sep)
    df.columns = df.columns.str.strip()
    return df

data_path = r'C:\Users\berkay\source\repos\customer-churn-prediction\data\customer_360_features.csv'
df = load_clean_csv(data_path)

print("Detected columns:", df.columns.tolist())
print("Is 'is_churned' in df?", 'is_churned' in df.columns)

Detected columns: ['customer_county', 'total_orders', 'total_items', 'total_spend', 'avg_order_value', 'avg_delivery_delay', 'max_delivery_delay', 'undelivered_count', 'sub_item_ratio', 'last_sub_plan', 'active_span_days', 'orders_per_month', 'undelivered_rate', 'is_churned', 'total_crm_tickets', 'complaint_count', 'query_count', 'fav_crm_platform', 'complaint_ratio', 'total_sessions', 'unique_ips', 'primary_device', 'nps_recommend', 'satisfaction_score', 'online_comparison_freq', 'income_min_k', 'sessions_per_order', 'complaints_per_order']
Is 'is_churned' in df? True


In [5]:
import pandas as pd

df = pd.read_parquet(r"C:\Users\berkay\source\repos\customer-churn-prediction\output\customer_360_features.parquet")
df.head()

,customer_county,total_orders,total_items,total_spend,avg_order_value,avg_delivery_delay,max_delivery_delay,undelivered_count,sub_item_ratio,last_sub_plan,...,complaint_ratio,total_sessions,unique_ips,primary_device,nps_recommend,satisfaction_score,online_comparison_freq,income_min_k,sessions_per_order,complaints_per_order
0,Berkshire,20,46,161096,4027.40,0.05,1.0,0,0.02,SuperSaver,...,0.67,48,48,Tablet,8,6,5,26.0,2.40,0.10
1,Oxfordshire,253,748,3110183,4748.37,0.04,3.0,0,0.02,SuperSaver,...,0.47,1085,1085,Tablet,5,8,9,34.0,4.29,0.08
2,Worcestershire,304,824,3409993,4789.32,0.04,3.0,0,0.02,Premium,...,0.54,624,624,Tablet,9,5,5,26.0,2.05,0.30
3,Kent,1055,7362,45511987,9291.95,0.09,4.0,0,0.00,Premium,...,0.47,1540,1540,Tablet,9,3,5,198.0,1.46,0.13
4,Hertfordshire,44,116,465845,4802.53,0.12,3.0,0,0.04,SuperSaver,...,0.20,388,388,Mobile,0,8,10,42.0,8.82,0.02


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.ml import Pipeline
from pyspark.ml.feature import StringIndexer, OneHotEncoder, VectorAssembler, StandardScaler
from pyspark.ml.classification import LogisticRegression, RandomForestClassifier, GBTClassifier
from pyspark.ml.evaluation import BinaryClassificationEvaluator, MulticlassClassificationEvaluator
from pyspark.ml.tuning import ParamGridBuilder, CrossValidator
from sklearn.metrics import confusion_matrix, roc_curve, auc, classification_report

# ==========================================
# 1. INITIALIZE SPARK SESSION
# ==========================================
spark = SparkSession.builder \
    .appName("PerfectPetClub_Churn_Prediction_Task3") \
    .master("local[*]") \
    .getOrCreate()

spark.sparkContext.setLogLevel("ERROR")

# ==========================================
# 2. LOAD PARQUET DATASET CREATED IN TASK 2
# ==========================================
parquet_path = r'C:\Users\berkay\source\repos\customer-churn-prediction\output\customer_360_features.parquet'
sdf = spark.read.parquet(parquet_path)

# PySpark MLlib expects target column as double named 'label'
sdf = sdf.withColumn("label", F.col("is_churned").cast("double")).drop("is_churned")

print(f"Loaded Parquet Dataset -> Rows: {sdf.count()} | Columns: {len(sdf.columns)}")
sdf.groupBy("label").count().show()

# ==========================================
# 3. FEATURE PREPROCESSING PIPELINE
# ==========================================
cat_cols = [c for c, dtype in sdf.dtypes if dtype == 'string']
num_cols = [c for c, dtype in sdf.dtypes if dtype != 'string' and c != 'label']

indexers = [
    StringIndexer(inputCol=c, outputCol=f"{c}_idx", handleInvalid="keep")
    for c in cat_cols
]
encoders = [
    OneHotEncoder(inputCol=f"{c}_idx", outputCol=f"{c}_ohe")
    for c in cat_cols
]

# Assemble numerical features separately so they can be scaled for Logistic Regression
num_assembler = VectorAssembler(inputCols=num_cols, outputCol="num_features", handleInvalid="keep")
scaler = StandardScaler(inputCol="num_features", outputCol="scaled_num_features", withStd=True, withMean=False)

final_inputs = ["scaled_num_features"] + [f"{c}_ohe" for c in cat_cols]
final_assembler = VectorAssembler(inputCols=final_inputs, outputCol="features", handleInvalid="keep")

preprocess_stages = indexers + encoders + [num_assembler, scaler, final_assembler]

# ==========================================
# 4. STRATIFIED TRAIN / TEST SPLIT (80% / 20%)
# ==========================================
train_0, test_0 = sdf.filter(F.col("label") == 0.0).randomSplit([0.8, 0.2], seed=42)
train_1, test_1 = sdf.filter(F.col("label") == 1.0).randomSplit([0.8, 0.2], seed=42)
train_df = train_0.union(train_1)
test_df = test_0.union(test_1)

print(f"Training Set: {train_df.count()} rows | Test Set: {test_df.count()} rows")

# ==========================================
# 5. MODEL SELECTION & HYPERPARAMETER GRIDS
# ==========================================
evaluator_auc = BinaryClassificationEvaluator(labelCol="label", metricName="areaUnderROC")
evaluator_f1 = MulticlassClassificationEvaluator(labelCol="label", predictionCol="prediction", metricName="f1")
evaluator_acc = MulticlassClassificationEvaluator(labelCol="label", predictionCol="prediction", metricName="accuracy")

# Model 1: Logistic Regression
lr = LogisticRegression(featuresCol="features", labelCol="label", maxIter=50)
lr_grid = ParamGridBuilder() \
    .addGrid(lr.regParam, [0.01, 0.1, 0.5]) \
    .addGrid(lr.elasticNetParam, [0.0, 0.5, 1.0]) \
    .build()

# Model 2: Random Forest Classifier
rf = RandomForestClassifier(featuresCol="features", labelCol="label", seed=42)
rf_grid = ParamGridBuilder() \
    .addGrid(rf.numTrees, [50, 100]) \
    .addGrid(rf.maxDepth, [3, 5]) \
    .build()

# Model 3: Gradient-Boosted Trees (GBTClassifier)
gbt = GBTClassifier(featuresCol="features", labelCol="label", seed=42)
gbt_grid = ParamGridBuilder() \
    .addGrid(gbt.maxIter, [20, 40]) \
    .addGrid(gbt.maxDepth, [2, 3]) \
    .addGrid(gbt.stepSize, [0.05, 0.1]) \
    .build()

models = {
    "Logistic Regression": (lr, lr_grid),
    "Random Forest": (rf, rf_grid),
    "GBT Classifier": (gbt, gbt_grid)
}

# ==========================================
# 6. HYPERPARAMETER TUNING (3-FOLD CROSS-VALIDATION)
# ==========================================
results = []
best_models = {}

for name, (clf, param_grid) in models.items():
    print(f"\n[TUNING] Training {name} with 3-Fold CrossValidator...")
    pipeline = Pipeline(stages=preprocess_stages + [clf])
    
    cv = CrossValidator(
        estimator=pipeline,
        estimatorParamMaps=param_grid,
        evaluator=evaluator_auc,
        numFolds=3,
        seed=42
    )
    
    cv_model = cv.fit(train_df)
    best_pipe_model = cv_model.bestModel
    best_models[name] = best_pipe_model
    
    preds = best_pipe_model.transform(test_df)
    test_auc = evaluator_auc.evaluate(preds)
    test_f1 = evaluator_f1.evaluate(preds)
    test_acc = evaluator_acc.evaluate(preds)
    
    results.append({
        "Model": name,
        "Best CV ROC-AUC": round(max(cv_model.avgMetrics), 3),
        "Test ROC-AUC": round(test_auc, 3),
        "Test F1-Score": round(test_f1, 3),
        "Test Accuracy": round(test_acc, 3)
    })

results_df = pd.DataFrame(results).sort_values(by="Test ROC-AUC", ascending=False)
print("\n=== PySpark Model Comparison Table ===")
print(results_df.to_string(index=False))

# ==========================================
# 7. EVALUATION PLOTS: CONFUSION MATRICES & ROC CURVES
# ==========================================
fig, axes = plt.subplots(1, 4, figsize=(20, 4.5))

for idx, (name, model) in enumerate(best_models.items()):
    preds_pd = model.transform(test_df).select("label", "prediction", "probability").toPandas()
    y_true = preds_pd["label"].values
    y_pred = preds_pd["prediction"].values
    y_prob = np.array([p[1] for p in preds_pd["probability"]])
    
    # Confusion Matrix
    cm = confusion_matrix(y_true, y_pred)
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', cbar=False, ax=axes[idx],
                xticklabels=['Active (0)', 'Churned (1)'],
                yticklabels=['Active (0)', 'Churned (1)'])
    axes[idx].set_title(f"{name}\nConfusion Matrix")
    axes[idx].set_xlabel("Predicted Label")
    axes[idx].set_ylabel("Actual Label")
    
    # ROC Curve
    fpr, tpr, _ = roc_curve(y_true, y_prob)
    roc_auc_val = auc(fpr, tpr)
    axes[3].plot(fpr, tpr, lw=2, label=f"{name} (AUC = {roc_auc_val:.2f})")

axes[3].plot([0, 1], [0, 1], color='gray', linestyle='--', label="Random Guess")
axes[3].set_title("ROC Curve Comparison")
axes[3].set_xlabel("False Positive Rate")
axes[3].set_ylabel("True Positive Rate")
axes[3].legend(loc="lower right")

plt.tight_layout()
plt.show()

# ==========================================
# 8. TOP 10 FEATURE IMPORTANCE (BEST TREE MODEL)
# ==========================================
rf_auc = results_df.loc[results_df["Model"] == "Random Forest", "Test ROC-AUC"].values[0]
gbt_auc = results_df.loc[results_df["Model"] == "GBT Classifier", "Test ROC-AUC"].values[0]
best_tree_name = "Random Forest" if rf_auc >= gbt_auc else "GBT Classifier"

best_tree_stage = best_models[best_tree_name].stages[-1]
transformed_sample = best_models[best_tree_name].transform(test_df)

# Extract human-readable feature names from Spark Vector metadata
attrs = transformed_sample.schema["features"].metadata["ml_attr"]["attrs"]
raw_names = []
for attr_type in ["numeric", "binary", "nominal"]:
    if attr_type in attrs:
        for attr in attrs[attr_type]:
            idx = attr["idx"]
            # Map scaled_num_features_0 back to original column name
            if attr["name"].startswith("scaled_num_features_"):
                num_idx = int(attr["name"].split("_")[-1])
                clean_name = num_cols[num_idx]
            else:
                clean_name = attr["name"]
            raw_names.append((idx, clean_name))

feat_names = [name for _, name in sorted(raw_names, key=lambda x: x[0])]

fi_df = pd.DataFrame({
    "Feature": feat_names,
    "Importance": best_tree_stage.featureImportances.toArray()
}).sort_values(by="Importance", ascending=False).head(10)

plt.figure(figsize=(9, 5))
plt.barh(fi_df["Feature"][::-1], fi_df["Importance"][::-1], color="#2a9d8f")
plt.xlabel("Relative Feature Importance")
plt.title(f"Top 10 Churn Drivers ({best_tree_name} - PySpark MLlib)")
plt.tight_layout()
plt.show()

# Print detailed classification report for the overall top-performing model
overall_best_name = results_df.iloc[0]["Model"]
best_preds_pd = best_models[overall_best_name].transform(test_df).select("label", "prediction").toPandas()
print(f"\n=== Detailed Classification Report ({overall_best_name}) ===")
print(classification_report(best_preds_pd["label"], best_preds_pd["prediction"], target_names=["Active (0)", "Churned (1)"]))

KeyboardInterrupt: 